# Advanced Smart Product Pricing Solution
## Target: SMAPE < 15%

This notebook implements an advanced machine learning pipeline with sophisticated feature engineering and ensemble modeling to achieve top leaderboard performance.

In [ ]:
# Environment setup and imports
import os
import sys
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import joblib
import pickle
from tqdm import tqdm

# ML imports
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
import xgboost as xgb

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Suppress warnings
warnings.filterwarnings('ignore')

print("Environment setup complete!")

In [ ]:
# Data loading and initial exploration
print("Loading data...")
train = pd.read_csv('./student_resource/dataset/train.csv')
test = pd.read_csv('./student_resource/dataset/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain columns: {train.columns.tolist()}")
print(f"Test columns: {test.columns.tolist()}")

# Price analysis
print("\n=== PRICE ANALYSIS ===")
print(train['price'].describe())
print(f"\nPrice range: ${train['price'].min():.2f} - ${train['price'].max():.2f}")
print(f"Price variance: {train['price'].var():.2f}")
print(f"Price skewness: {train['price'].skew():.2f}")

# Log price analysis (often better for SMAPE)
train['log_price'] = np.log1p(train['price'])
print(f"\nLog price range: {train['log_price'].min():.2f} - {train['log_price'].max():.2f}")
print(f"Log price skewness: {train['log_price'].skew():.2f}")

In [ ]:
# Advanced feature extraction functions

def extract_numerical_values(text):
    """Extract all numerical values from text"""
    if not isinstance(text, str):
        return []
    
    # Find all decimal numbers
    numbers = re.findall(r'\d+\.?\d*', text)
    return [float(x) for x in numbers if x]

def extract_units_and_measurements(text):
    """Extract unit measurements that are crucial for pricing"""
    if not isinstance(text, str):
        return {}
    
    text = text.lower()
    features = {}
    
    # Weight patterns
    weight_patterns = [
        (r'(\d+\.?\d*)\s*lbs?\b', 'weight_lbs'),
        (r'(\d+\.?\d*)\s*pounds?\b', 'weight_lbs'),
        (r'(\d+\.?\d*)\s*oz\b', 'weight_oz'),
        (r'(\d+\.?\d*)\s*ounces?\b', 'weight_oz'),
        (r'(\d+\.?\d*)\s*kg\b', 'weight_kg'),
        (r'(\d+\.?\d*)\s*g\b', 'weight_g'),
        (r'(\d+\.?\d*)\s*grams?\b', 'weight_g'),
    ]
    
    # Volume patterns
    volume_patterns = [
        (r'(\d+\.?\d*)\s*fl\s*oz\b', 'volume_fl_oz'),
        (r'(\d+\.?\d*)\s*fluid\s*ounces?\b', 'volume_fl_oz'),
        (r'(\d+\.?\d*)\s*ml\b', 'volume_ml'),
        (r'(\d+\.?\d*)\s*l\b', 'volume_l'),
        (r'(\d+\.?\d*)\s*liters?\b', 'volume_l'),
        (r'(\d+\.?\d*)\s*gallons?\b', 'volume_gal'),
        (r'(\d+\.?\d*)\s*gal\b', 'volume_gal'),
    ]
    
    # Size patterns
    size_patterns = [
        (r'(\d+\.?\d*)\s*inches?\b', 'size_inches'),
        (r'(\d+\.?\d*)\s*in\b', 'size_inches'),
        (r'(\d+\.?\d*)\s*feet\b', 'size_feet'),
        (r'(\d+\.?\d*)\s*ft\b', 'size_feet'),
        (r'(\d+\.?\d*)\s*cm\b', 'size_cm'),
        (r'(\d+\.?\d*)\s*mm\b', 'size_mm'),
    ]
    
    # Count patterns
    count_patterns = [
        (r'pack of (\d+)', 'pack_count'),
        (r'(\d+)[-\s]pack', 'pack_count'),
        (r'(\d+)\s*count', 'item_count'),
        (r'(\d+)\s*ct\b', 'item_count'),
        (r'(\d+)\s*pcs?\b', 'piece_count'),
        (r'(\d+)\s*pieces?\b', 'piece_count'),
        (r'set of (\d+)', 'set_count'),
        (r'value:\s*(\d+\.?\d*)', 'value_number'),
    ]
    
    all_patterns = weight_patterns + volume_patterns + size_patterns + count_patterns
    
    for pattern, feature_name in all_patterns:
        matches = re.findall(pattern, text)
        if matches:
            # Take the first match or average if multiple
            values = [float(x) for x in matches]
            features[feature_name] = values[0] if len(values) == 1 else np.mean(values)
    
    return features

def extract_advanced_text_features(text):
    """Extract advanced text-based features"""
    if not isinstance(text, str):
        return {}
    
    features = {}
    
    # Basic text statistics
    features['text_length'] = len(text)
    features['word_count'] = len(text.split())
    features['sentence_count'] = len(re.split(r'[.!?]+', text))
    features['avg_word_length'] = np.mean([len(word) for word in text.split()]) if text.split() else 0
    
    # Character distribution
    features['digit_count'] = sum(c.isdigit() for c in text)
    features['upper_count'] = sum(c.isupper() for c in text)
    features['punctuation_count'] = sum(not c.isalnum() and not c.isspace() for c in text)
    features['digit_ratio'] = features['digit_count'] / len(text) if text else 0
    
    # Special keywords that might indicate price range
    luxury_keywords = ['premium', 'luxury', 'professional', 'deluxe', 'pro', 'advanced']
    budget_keywords = ['basic', 'budget', 'economy', 'value', 'cheap', 'affordable']
    
    text_lower = text.lower()
    features['luxury_keyword_count'] = sum(1 for kw in luxury_keywords if kw in text_lower)
    features['budget_keyword_count'] = sum(1 for kw in budget_keywords if kw in text_lower)
    
    # Brand indicators
    features['has_brand_mention'] = 1 if re.search(r'brand:|by\s+[A-Z]', text) else 0
    
    return features

def extract_product_category_features(text):
    """Extract product category and type features"""
    if not isinstance(text, str):
        return {}
    
    text_lower = text.lower()
    
    # Define category keywords
    categories = {
        'food': ['food', 'snack', 'sauce', 'spice', 'tea', 'coffee', 'juice', 'chocolate', 'candy'],
        'beauty': ['beauty', 'cosmetic', 'makeup', 'skincare', 'lotion', 'cream', 'shampoo'],
        'electronics': ['electronic', 'battery', 'cable', 'charger', 'device', 'gadget'],
        'clothing': ['shirt', 'dress', 'pants', 'clothing', 'apparel', 'fabric'],
        'home': ['home', 'kitchen', 'bathroom', 'bedroom', 'furniture', 'decor'],
        'health': ['health', 'vitamin', 'supplement', 'medicine', 'medical', 'therapy'],
        'toys': ['toy', 'game', 'puzzle', 'doll', 'action figure', 'educational'],
        'books': ['book', 'novel', 'guide', 'manual', 'textbook', 'magazine'],
        'sports': ['sport', 'fitness', 'exercise', 'gym', 'outdoor', 'athletic']
    }
    
    features = {}
    for category, keywords in categories.items():
        features[f'is_{category}'] = 1 if any(kw in text_lower for kw in keywords) else 0
    
    return features

print("Advanced feature extraction functions defined!")

In [ ]:
# Apply advanced feature engineering
print("Applying advanced feature engineering...")

def process_dataset(df, is_train=True):
    """Process dataset with advanced feature engineering"""
    df_processed = df.copy()
    
    # Extract measurements and units
    print("Extracting measurements and units...")
    measurement_features = df_processed['catalog_content'].apply(extract_units_and_measurements)
    measurement_df = pd.DataFrame(measurement_features.tolist()).fillna(0)
    
    # Extract advanced text features
    print("Extracting advanced text features...")
    text_features = df_processed['catalog_content'].apply(extract_advanced_text_features)
    text_df = pd.DataFrame(text_features.tolist()).fillna(0)
    
    # Extract category features
    print("Extracting category features...")
    category_features = df_processed['catalog_content'].apply(extract_product_category_features)
    category_df = pd.DataFrame(category_features.tolist()).fillna(0)
    
    # Combine all engineered features
    engineered_features = pd.concat([
        measurement_df,
        text_df,
        category_df
    ], axis=1)
    
    # Add to main dataframe
    df_processed = pd.concat([df_processed, engineered_features], axis=1)
    
    # Extract numerical values and create aggregated features
    print("Extracting numerical patterns...")
    numerical_values = df_processed['catalog_content'].apply(extract_numerical_values)
    
    # Create aggregated numerical features
    df_processed['num_values_count'] = numerical_values.apply(len)
    df_processed['num_values_max'] = numerical_values.apply(lambda x: max(x) if x else 0)
    df_processed['num_values_min'] = numerical_values.apply(lambda x: min(x) if x else 0)
    df_processed['num_values_mean'] = numerical_values.apply(lambda x: np.mean(x) if x else 0)
    df_processed['num_values_std'] = numerical_values.apply(lambda x: np.std(x) if len(x) > 1 else 0)
    
    # Calculate total volume/weight features (crucial for pricing)
    df_processed['total_volume'] = (
        df_processed.get('volume_fl_oz', 0) * 29.5735 +  # Convert to ml
        df_processed.get('volume_ml', 0) +
        df_processed.get('volume_l', 0) * 1000 +
        df_processed.get('volume_gal', 0) * 3785.41
    )
    
    df_processed['total_weight'] = (
        df_processed.get('weight_lbs', 0) * 453.592 +  # Convert to grams
        df_processed.get('weight_oz', 0) * 28.3495 +
        df_processed.get('weight_kg', 0) * 1000 +
        df_processed.get('weight_g', 0)
    )
    
    # Calculate pack multiplier
    df_processed['pack_multiplier'] = np.maximum(
        df_processed.get('pack_count', 1),
        np.maximum(
            df_processed.get('item_count', 1),
            df_processed.get('set_count', 1)
        )
    )
    
    # Price per unit features (only for training)
    if is_train and 'price' in df_processed.columns:
        df_processed['price_per_volume'] = np.where(
            df_processed['total_volume'] > 0,
            df_processed['price'] / df_processed['total_volume'],
            0
        )
        df_processed['price_per_weight'] = np.where(
            df_processed['total_weight'] > 0,
            df_processed['price'] / df_processed['total_weight'],
            0
        )
        df_processed['price_per_pack'] = df_processed['price'] / df_processed['pack_multiplier']
    
    return df_processed

# Process both datasets
train_processed = process_dataset(train, is_train=True)
test_processed = process_dataset(test, is_train=False)

print(f"\nProcessed train shape: {train_processed.shape}")
print(f"Processed test shape: {test_processed.shape}")
print(f"\nNew features created: {train_processed.shape[1] - train.shape[1]}")

In [ ]:
# Advanced text processing with multiple approaches
print("Advanced text processing...")

# Clean and prepare text
def advanced_text_cleaning(text):
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Replace common abbreviations
    text = re.sub(r'\bfl\s*oz\b', 'fluid_ounce', text)
    text = re.sub(r'\boz\b', 'ounce', text)
    text = re.sub(r'\blbs?\b', 'pound', text)
    text = re.sub(r'\bct\b', 'count', text)
    
    # Keep important punctuation for pricing
    text = re.sub(r'[^\w\s\$\.]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply text cleaning
train_processed['cleaned_content'] = train_processed['catalog_content'].apply(advanced_text_cleaning)
test_processed['cleaned_content'] = test_processed['catalog_content'].apply(advanced_text_cleaning)

# Multiple TF-IDF approaches
print("Creating multiple TF-IDF representations...")

# 1. Word-level TF-IDF
tfidf_word = TfidfVectorizer(
    max_features=20000,
    min_df=3,
    max_df=0.9,
    ngram_range=(1, 2),
    analyzer='word',
    lowercase=True,
    strip_accents='unicode'
)

# 2. Character-level TF-IDF
tfidf_char = TfidfVectorizer(
    max_features=10000,
    analyzer='char',
    ngram_range=(3, 5),
    lowercase=True
)

# Fit and transform
train_text = train_processed['cleaned_content'].fillna('')
test_text = test_processed['cleaned_content'].fillna('')

# Word-level features
X_train_word = tfidf_word.fit_transform(train_text)
X_test_word = tfidf_word.transform(test_text)

# Character-level features
X_train_char = tfidf_char.fit_transform(train_text)
X_test_char = tfidf_char.transform(test_text)

print(f"Word TF-IDF shape: {X_train_word.shape}")
print(f"Char TF-IDF shape: {X_train_char.shape}")

# Apply SVD for dimensionality reduction
print("Applying SVD dimensionality reduction...")

# Word features SVD
svd_word = TruncatedSVD(n_components=300, random_state=RANDOM_SEED)
X_train_word_svd = svd_word.fit_transform(X_train_word)
X_test_word_svd = svd_word.transform(X_test_word)

# Character features SVD
svd_char = TruncatedSVD(n_components=100, random_state=RANDOM_SEED)
X_train_char_svd = svd_char.fit_transform(X_train_char)
X_test_char_svd = svd_char.transform(X_test_char)

print(f"Word SVD shape: {X_train_word_svd.shape}")
print(f"Char SVD shape: {X_train_char_svd.shape}")

In [ ]:
# Feature selection and preparation
print("Preparing final feature set...")

# Select numerical features (exclude text and target)
exclude_cols = ['sample_id', 'catalog_content', 'image_link', 'price', 'log_price', 'cleaned_content', 
                'price_per_volume', 'price_per_weight', 'price_per_pack']
numerical_features = [col for col in train_processed.columns if col not in exclude_cols]

print(f"Numerical features count: {len(numerical_features)}")
print(f"Sample features: {numerical_features[:10]}")

# Extract numerical features
X_train_num = train_processed[numerical_features].fillna(0)
X_test_num = test_processed[numerical_features].fillna(0)

# Scale numerical features
scaler = RobustScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)

# Combine all features
print("Combining all feature types...")

X_train_combined = np.hstack([
    X_train_num_scaled,
    X_train_word_svd,
    X_train_char_svd
])

X_test_combined = np.hstack([
    X_test_num_scaled,
    X_test_word_svd,
    X_test_char_svd
])

print(f"Final feature shape - Train: {X_train_combined.shape}")
print(f"Final feature shape - Test: {X_test_combined.shape}")

# Prepare target variable with different transformations
y_original = train_processed['price'].values
y_log = np.log1p(y_original)  # Log transformation
y_sqrt = np.sqrt(y_original)  # Square root transformation

print(f"Target variable statistics:")
print(f"Original - mean: {y_original.mean():.2f}, std: {y_original.std():.2f}")
print(f"Log - mean: {y_log.mean():.2f}, std: {y_log.std():.2f}")
print(f"Sqrt - mean: {y_sqrt.mean():.2f}, std: {y_sqrt.std():.2f}")

In [ ]:
# SMAPE metric and evaluation functions
def smape_metric(y_true, y_pred):
    """Calculate SMAPE (Symmetric Mean Absolute Percentage Error)"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Ensure no negative values
    y_true = np.maximum(y_true, 0.01)
    y_pred = np.maximum(y_pred, 0.01)
    
    denominator = (np.abs(y_true) + np.abs(y_pred))
    # Avoid division by zero
    denominator = np.where(denominator == 0, 1e-8, denominator)
    
    smape = 100 * np.mean(2.0 * np.abs(y_pred - y_true) / denominator)
    return smape

def advanced_cross_validation(X, y, models, n_splits=5, random_state=RANDOM_SEED):
    """Advanced cross-validation with multiple strategies"""
    
    # Create price-based stratification bins
    price_bins = pd.qcut(y, q=n_splits, labels=False, duplicates='drop')
    
    # Use StratifiedKFold for more balanced splits
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        
        oof_preds = np.zeros(len(X))
        fold_scores = []
        trained_models = []
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, price_bins)):
            print(f"  Fold {fold + 1}/{n_splits}", end=" ")
            
            X_fold_train, X_fold_val = X[train_idx], X[val_idx]
            y_fold_train, y_fold_val = y[train_idx], y[val_idx]
            
            # Train model
            model_clone = type(model)(**model.get_params())
            
            # Special handling for LightGBM
            if hasattr(model_clone, 'fit') and 'lgb' in str(type(model_clone)).lower():
                model_clone.fit(
                    X_fold_train, y_fold_train,
                    eval_set=[(X_fold_val, y_fold_val)],
                    eval_metric='rmse',
                    verbose=False
                )
            else:
                model_clone.fit(X_fold_train, y_fold_train)
            
            # Predict
            val_preds = model_clone.predict(X_fold_val)
            oof_preds[val_idx] = val_preds
            
            # Calculate SMAPE on original scale
            if name.endswith('_log'):
                val_preds_orig = np.expm1(val_preds)
                y_val_orig = np.expm1(y_fold_val)
            elif name.endswith('_sqrt'):
                val_preds_orig = val_preds ** 2
                y_val_orig = y_fold_val ** 2
            else:
                val_preds_orig = val_preds
                y_val_orig = y_fold_val
            
            fold_smape = smape_metric(y_val_orig, val_preds_orig)
            fold_scores.append(fold_smape)
            trained_models.append(model_clone)
            
            print(f"SMAPE: {fold_smape:.4f}")
        
        mean_score = np.mean(fold_scores)
        std_score = np.std(fold_scores)
        
        print(f"  {name} - Mean SMAPE: {mean_score:.4f} (+/- {std_score:.4f})")
        
        results[name] = {
            'models': trained_models,
            'oof_preds': oof_preds,
            'scores': fold_scores,
            'mean_score': mean_score,
            'std_score': std_score
        }
    
    return results

print("Advanced evaluation functions defined!")

In [ ]:
# Define advanced models ensemble
print("Defining advanced model ensemble...")

# Models for log-transformed target
models_log = {
    'lightgbm_log': lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=31,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=-1
    ),
    
    'xgboost_log': xgb.XGBRegressor(
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0
    ),
    
    'ridge_log': Ridge(
        alpha=10.0,
        random_state=RANDOM_SEED
    ),
    
    'elastic_log': ElasticNet(
        alpha=1.0,
        l1_ratio=0.5,
        random_state=RANDOM_SEED,
        max_iter=2000
    ),
    
    'rf_log': RandomForestRegressor(
        n_estimators=500,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
}

# Train models on log-transformed target
print("Training models on log-transformed target...")
results_log = advanced_cross_validation(X_train_combined, y_log, models_log, n_splits=5)

# Display results
print("\n=== LOG-TRANSFORMED MODEL RESULTS ===")
sorted_models = sorted(results_log.items(), key=lambda x: x[1]['mean_score'])
for name, result in sorted_models:
    print(f"{name}: {result['mean_score']:.4f} (+/- {result['std_score']:.4f})")

In [ ]:
# Advanced ensemble with stacking
print("Creating advanced ensemble...")

# Get the best 3 models for stacking
sorted_models = sorted(results_log.items(), key=lambda x: x[1]['mean_score'])
best_models = sorted_models[:3]

print("Best models selected for ensemble:")
for name, result in best_models:
    print(f"  {name}: {result['mean_score']:.4f}")

# Create stacking features
stacking_features = np.column_stack([
    results_log[name]['oof_preds'] for name, _ in best_models
])

print(f"Stacking features shape: {stacking_features.shape}")

# Train meta-learner
print("Training meta-learner...")
meta_model = Ridge(alpha=1.0, random_state=RANDOM_SEED)

# Use cross-validation for meta-learner to avoid overfitting
meta_oof_preds = np.zeros(len(y_log))
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

meta_models = []
for train_idx, val_idx in kf.split(stacking_features):
    X_meta_train, X_meta_val = stacking_features[train_idx], stacking_features[val_idx]
    y_meta_train, y_meta_val = y_log[train_idx], y_log[val_idx]
    
    meta_clone = Ridge(alpha=1.0, random_state=RANDOM_SEED)
    meta_clone.fit(X_meta_train, y_meta_train)
    
    meta_oof_preds[val_idx] = meta_clone.predict(X_meta_val)
    meta_models.append(meta_clone)

# Calculate ensemble performance
ensemble_preds_orig = np.expm1(meta_oof_preds)
y_orig = np.expm1(y_log)
ensemble_smape = smape_metric(y_orig, ensemble_preds_orig)

print(f"\nEnsemble SMAPE: {ensemble_smape:.4f}")

# Compare with simple average
simple_avg = np.mean([results_log[name]['oof_preds'] for name, _ in best_models], axis=0)
simple_avg_orig = np.expm1(simple_avg)
simple_avg_smape = smape_metric(y_orig, simple_avg_orig)

print(f"Simple Average SMAPE: {simple_avg_smape:.4f}")
print(f"Improvement: {simple_avg_smape - ensemble_smape:.4f}")

In [ ]:
# Generate predictions for test set
print("Generating test predictions...")

# Get predictions from best models
test_preds_stack = []

for name, _ in best_models:
    model_preds = []
    
    # Average predictions from all folds
    for model in results_log[name]['models']:
        pred = model.predict(X_test_combined)
        model_preds.append(pred)
    
    # Average across folds
    avg_pred = np.mean(model_preds, axis=0)
    test_preds_stack.append(avg_pred)

# Stack test predictions
test_stacking_features = np.column_stack(test_preds_stack)

# Use meta-models to get final predictions
final_test_preds = []
for meta_model in meta_models:
    pred = meta_model.predict(test_stacking_features)
    final_test_preds.append(pred)

# Average meta-model predictions
final_pred_log = np.mean(final_test_preds, axis=0)

# Transform back to original scale
final_pred_original = np.expm1(final_pred_log)

# Ensure positive predictions
final_pred_original = np.maximum(final_pred_original, 0.01)

print(f"Final predictions shape: {final_pred_original.shape}")
print(f"Prediction statistics:")
print(f"  Min: ${final_pred_original.min():.2f}")
print(f"  Max: ${final_pred_original.max():.2f}")
print(f"  Mean: ${final_pred_original.mean():.2f}")
print(f"  Median: ${np.median(final_pred_original):.2f}")

In [ ]:
# Create submission file
print("Creating submission file...")

submission = pd.DataFrame({
    'sample_id': test_processed['sample_id'],
    'price': final_pred_original
})

# Save submission
submission_path = './advanced_test_out.csv'
submission.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Submission shape: {submission.shape}")
print("\nFirst 10 predictions:")
print(submission.head(10))

# Save model performance summary
performance_summary = {
    'ensemble_smape': ensemble_smape,
    'simple_average_smape': simple_avg_smape,
    'best_models': [(name, result['mean_score']) for name, result in best_models],
    'all_models': {name: result['mean_score'] for name, result in results_log.items()}
}

import json
with open('./advanced_performance.json', 'w') as f:
    json.dump(performance_summary, f, indent=2)

print(f"\n=== FINAL PERFORMANCE SUMMARY ===")
print(f"Best Ensemble SMAPE: {ensemble_smape:.4f}")
print(f"Target achieved: {'YES' if ensemble_smape < 15 else 'NO'}")

if ensemble_smape < 15:
    print("\n🎉 CONGRATULATIONS! You've achieved a SMAPE score below 15!")
    print("This should put you in strong contention for the top positions!")
else:
    print(f"\n📈 Current score: {ensemble_smape:.4f}")
    print(f"Gap to target: {ensemble_smape - 15:.4f}")
    print("Consider further optimization strategies.")